[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs_ar/Lab_5_NonHomogeneous_PDEs_AR.ipynb)


In [ ]:
# 🚀 تهيئة بيئة Google Colab
# إذا كنت تشغل هذا النوت بوك على Google Colab، شغل هذه الخلية لتحميل ملفات المستودع والاختبارات تلقائياً:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] تم تجهيز بيئة Google Colab بنجاح والاتصال بمستودع المشروع!')
except NameError:
    pass


<div dir="rtl" style="text-align:right">

# 📡 اللاب 5: المعادلات الخطية غير المتجانسة في الرتبة، تحليل المؤثرات، واضمحلال الجهد في الممرات الضيقة
### *غرفة التحكم: لماذا يفشل شرط ديريكليه (`dirichlet_potential`) في الممرات الضيقة بسبب الاضمحلال الأسي؟*
**المقرر:** المعادلات التفاضلية الجزئية وتطبيقاتها في الملاحة الفضائية (Applied PDE & Planetary Robotics)  
**المرجع في المذكرة:** الباب الثاني (Chapter 2)، القسم 2.1 — **ترقيم المذكرة (الباب الثاني) ص 1–10** *(صفحات ملف الـ PDF من 52 إلى 62؛ قانون العوامل الخطية في ص 2 / PDF ص 54، والمثال 5 في ص 4 / PDF ص 56)*

---
### 🗺️ موقعك في سلسلة بناء محرك الملاحة (Rover Pipeline)
```
[اللاب 1–3: الميول، مسارات الانسياب، وهوامش الأمان] ──► [اللاب 4: معادلة لابلاس] ──► 📍 [اللاب 5: المؤثرات غير المتجانسة واضمحلال الجهد] ──► [اللاب 6: محرك الملاحة النهائي SOR + Neumann]
```

---
### 🧭 حجر رشيد (The Rosetta Stone): الربط بين السبورة ومحرك المركبة

| 1. رمز المذكرة (PDF pp. 52–62) | 2. المعنى الهندسي والفيزيائي | 3. الدالة الفعلية في المشروع (`rover_path_finder/`) |
| :--- | :--- | :--- |
| $F(D_x, D_y)z = 0$ غير متجانسة في الرتبة | معادلة تجمع مشتقات من رتب مختلفة أو حداً ثابتاً $c$ (مثل $aD_x + bD_y + c$) | تمثيل التخميد والاضمحلال الأسي على طول مسار الانتشار |
| $(a D_x + b D_y + c)z = 0 \implies z = e^{-\frac{c}{a}x}\phi(ay - bx)$ | كل عامل خطي به ثابت $c \neq 0$ يفرض غلافاً أسياً $e^{-(c/a)x}$ على الحل | الاضمحلال الأسي للانحدار داخل الممرات الضيقة |
| $(F_1 \cdot F_2)z = 0 \implies z = z_1 + z_2$ | تحليل المؤثر إلى عوامل خطية وجمع الحلول (مبدأ التراكب) | تحليل أنماط الجهد إلى مركبات أسية متراكبة |
| تلاشي انحدار الجهد $\|\nabla\phi\| \propto e^{-\pi d / w}$ | عندما نجعل جدران العوائق ثابتة عند $\phi=1$، يتلاشى الميل أسياً داخل أي ممر عرضه $w$ | سبب استبدال [`planners.dirichlet_potential`](rover_path_finder/planners.py) بالدالة [`planners.neumann_potential`](rover_path_finder/planners.py) |

---
## 1. أولاً: توضيح مصطلح مهم جداً في المذكرة ("غير متجانسة" في الباب الثاني!)

عندما تقرأ عنوان **الباب الثاني في المذكرة (PDF ص 52): "Non-Homogeneous Linear PDEs"**، انتبه جيداً للفرق الاصطلاحي:
* في المعادلات العادية (ODEs)، كلمة "غير متجانسة" تعني عادةً وجود طرف أيمن $f(x, y) \neq 0$.
* أما في **الباب الثاني من مذكرتنا (PDF ص 52–57)**، كلمة **Non-Homogeneous (غير متجانسة في الرتبة)** تعني أن **الطرف الأيسر نفسه يحتوي على مشتقات من رتب مختلفة** (مثلاً مشتقة ثانية $D_x^2$ مع مشتقة أولى $D_x$ أو ثابت $c$)، مثل:
  $$(2 D_x - 3 D_y + 1)(D_x + 2 D_y - 2)z = 0$$
  لاحظ أن الطرف الأيمن يساوي صفراً، ومع ذلك تسمى المعادلة **غير متجانسة في الرتبة** بسبب وجود الثابتين $+1$ و $-2$ مع المشتقات الأولى $D_x, D_y$!

---
## 2. كيف يرتبط هذا الاضمحلال الأسي بمشكلة الممرات الضيقة في مركبة المريخ؟

في ملف [`rover_path_finder/planners.py`](rover_path_finder/planners.py)، توجد طريقتان لبناء حقل الجهد التوافقي للمركبة:
1. **الصياغة الأولى (`dirichlet_potential` — Formulation A):** تثبيت جميع العوائق وجدران الخريطة عند جهد مرتفع ثابت $\phi = 1$، والهدف عند $\phi = 0$.
2. **الصياغة الثانية (`neumann_potential` — Formulation B):** جعل العوائق جدراناً عازلة ($\partial\phi/\partial n = 0$) ووضع مصدر عند البداية ($\phi=1$) ومصب عند الهدف ($\phi=0$).

**لماذا تفشل الصياغة الأولى (`dirichlet_potential`) عندما تمر المركبة في ممر ضيق بين صخرتين؟**
إذا دخلت المركبة ممراً ضيقاً عرضه $w$ وطوله $d$ وجدرانه مثبتة عند $\phi = 1$، فإن فرق الجهد $\tilde{\phi} = 1 - \phi$ يحقق نمطاً توافقياً في اتجاه العرض $\sin(\pi y / w)$ مما يفرض معادلة عاملية ذات حد ثابت $c = \pm\frac{\pi}{w}$ في اتجاه الطول $x$:
$$\left(D_x + \frac{\pi}{w}\right)\tilde{\phi} = 0 \implies \tilde{\phi}(x, y) \propto e^{-\frac{\pi}{w} x} \sin\left(\frac{\pi y}{w}\right)$$
أي أن انحدار الجهد يتلاشى أسياً بمعدل $e^{-\pi x / w}$! فإذا كان الممر ضيقاً ($w$ صغير)، يصبح الجهد $\phi \approx 1.000000$ في كل مكان داخل الممر ويتوقف انحدار المركبة تماماً!

</div>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab5

%matplotlib inline
print("تم تهيئة بيئة اللاب 5 بنجاح!")


<div dir="rtl" style="text-align:right">

---
## 3. الوصفة الرياضية الكاملة للباب الثاني في المذكرة (Cases A, B, C, D & P.I. — PDF ص 52–62)

### الحالة (A): عامل خطي واحد أو مكرر $(a D_x + b D_y + c)^k z = 0$ *(الباب الثاني ص 2–3 / PDF ص 54–55)*

#### 1. لماذا يظهر العامل الأسي $e^{-\frac{c}{a}x}$؟
* المعادلة $(a D_x + b D_y + c)z = 0$ هي $a \frac{\partial z}{\partial x} + b \frac{\partial z}{\partial y} = -c z$ (معادلة لاجرانج من اللاب 2 حيث $P=a, Q=b, R=-cz$!).
* من معادلات لاجرانج المساعدة:
  $$\frac{dx}{a} = \frac{dy}{b} = \frac{dz}{-cz}$$
* من الكسر الأول والثاني: $b\,dx - a\,dy = 0 \implies ay - bx = u$.
* من الكسر الأول والثالث (بشرط $a \neq 0$): $\frac{dz}{z} = -\frac{c}{a}dx \implies \ln|z| = -\frac{c}{a}x + \ln|\phi(u)|$.
* إذن **القانون المباشر لأي عامل خطي** هو:
  * **إذا كان $a \neq 0$:**
    $$\boxed{(a D_x + b D_y + c)z = 0 \implies z = e^{-\frac{c}{a}x}\,\phi(a y - b x)}$$
  * **وإذا كان $a = 0$ و $b \neq 0$ (أو إذا اخترنا الكسر الثاني والثالث $\frac{dy}{b} = \frac{dz}{-cz}$):**
    $$\boxed{z = e^{-\frac{c}{b}y}\,\phi(a y - b x)}$$

#### 2. ماذا لو تكرر العامل الخطي $k$ من المرات؟ $(a D_x + b D_y + c)^k z = 0$ (ص 2–3 / PDF ص 54–55)
* **إذا استخدمنا الأس بدلالة $x$ ($a \neq 0$):** نضرب الدوال المتتالية في $1, x, x^2, \dots, x^{k-1}$:
  $$\boxed{z = e^{-\frac{c}{a}x}\left[\phi_1(ay - bx) + x\,\phi_2(ay - bx) + x^2\,\phi_3(ay - bx) + \dots + x^{k-1}\phi_k(ay - bx)\right]}$$
* **وإذا استخدمنا الأس بدلالة $y$ ($b \neq 0$، خاصة عندما $a = 0$):** نضرب في $1, y, y^2, \dots, y^{k-1}$:
  $$\boxed{z = e^{-\frac{c}{b}y}\left[\phi_1(ay - bx) + y\,\phi_2(ay - bx) + \dots + y^{k-1}\phi_k(ay - bx)\right]}$$

> 📌 **تنبيه خطأ مطبعي مهم جداً في المذكرة (الباب الثاني ص 2–3، المثالان 3 و 4 / PDF ص 54–55):**
> * في **المثال 3 (PDF ص 54):** $(2D_x - 3D_y + 2)^2 z = 0$ ($a=2, b=-3, c=2$)، الصيغة الأولى في المذكرة $z = e^{-x}[\phi_1(2y+3x) + x\phi_2(2y+3x)]$ **صحيحة 100%**! لكن في السطر الذي يليه ("Or $z = e^{\frac{2}{3}y}[\dots]$")، كتبت المذكرة $x\phi_2$ بدلاً من $y\phi_2$.
> * وفي **المثال 4 (PDF ص 55):** $(D_x - 2D_y + 1)^4 z = 0$ ($a=1, b=-2, c=1, k=4$)، وضعت المذكرة داخل القوس قوى $x$ ($1, x, x^2, x^3$) ولكنها كتبت خارج القوس بالخطأ $e^{\frac{1}{2}y}$ بدلاً من $e^{-\frac{c}{a}x} = e^{-x}$!
> * **الخلاصة الذهبية لك في الامتحان:** التزم دائماً بصيغة $x$ القياسية (طالما $a \neq 0$): **أخرج $e^{-(c/a)x}$ واضرب بالداخل في قوى $x$ ($1, x, x^2, \dots$)**، فهي الأسهل والأضمن 100%!

---
### الحالة (B): المؤثرات القابلة للتحليل إلى عوامل خطية مختلفة *(الباب الثاني ص 3–4 / PDF ص 55–56)*
إذا أمكن تحليل المؤثر $F(D_x, D_y)$ إلى حاصل ضرب عوامل خطية:
$$(a_1 D_x + b_1 D_y + c_1)(a_2 D_x + b_2 D_y + c_2)z = 0$$
فإن الحل العام هو مجموع حلول كل عامل على حدة:
$$z = e^{-\frac{c_1}{a_1}x}\phi_1(a_1 y - b_1 x) + e^{-\frac{c_2}{a_2}x}\phi_2(a_2 y - b_2 x)$$

#### 📖 مثال محلول بالتفصيل (المثال 5، الباب الثاني ص 4 / PDF ص 56)
**المسألة:** أوجد الحل العام للمعادلة:
$$(2 D_x - 3 D_y + 1)(D_x + 2 D_y - 2)z = 0$$
1. **العامل الأول $(2 D_x - 3 D_y + 1)z = 0$:**
   * المعاملات: $a_1 = 2,\; b_1 = -3,\; c_1 = 1$.
   * الأس: $-\frac{c_1}{a_1}x = -\frac{1}{2}x$.
   * داخل الدالة الاختيارية: $a_1 y - b_1 x = 2y - (-3)x = 2y + 3x$.
   * إذن الحل الأول: $z_1 = e^{-x/2}\Phi_1(2y + 3x)$.
2. **العامل الثاني $(D_x + 2 D_y - 2)z = 0$:**
   * المعاملات: $a_2 = 1,\; b_2 = 2,\; c_2 = -2$.
   * الأس: $-\frac{c_2}{a_2}x = -\frac{-2}{1}x = 2x$.
   * داخل الدالة الاختيارية: $a_2 y - b_2 x = 1y - 2x = y - 2x$.
   * إذن الحل الثاني: $z_2 = e^{2x}\Phi_2(y - 2x)$.
3. **الحل العام الكلي:** $\boxed{z = e^{-x/2}\Phi_1(2y + 3x) + e^{2x}\Phi_2(y - 2x)}$.

*(وكذلك في **المثال 7 ص 5 / PDF ص 57**: المعادلة $(D_x^2 - D_y^2 - 3D_x + 3D_y)z = 0$ نحللها بالتجميع أولاً: $(D_x - D_y)(D_x + D_y) - 3(D_x - D_y) = (D_x - D_y)(D_x + D_y - 3)z = 0 \implies \boxed{z = \phi_1(y + x) + e^{3x}\phi_2(y - x)}$!)*

---
### الحالتان (C) و (D): المؤثرات غير القابلة للتحليل (Irreducible Operators — الباب الثاني ص 5–8 / PDF ص 57–60)
* **متى نستخدم الحالة (C)؟** عندما لا يمكن تحليل $F(D_x, D_y)$ إلى عوامل خطية من الدرجة الأولى، مثل $(D_x - D_y^3)z = 0$ أو $(D_x^2 + D_x + D_y)z = 0$ أو معادلة الحرارة $(D_x^2 - D_y)z = 0$.
* **قاعدة الحل في المذكرة (ص 5–6 / PDF ص 57–58):**
  نفرض الحل في صورة متسلسلة من الدوال الأسية:
  $$\boxed{z = \sum_{i=1}^{\infty} A_i\, e^{a_i x + b_i y} \qquad \text{بشرط أن تحقق } (a_i, b_i)\text{ المعادلة المميزة: } F(a_i, b_i) = 0}$$
  حيث $A_i$ و $a_i$ (أو $b_i$) ثوابت اختيارية!
* **📖 الأمثلة المحلولة في المذكرة (أمثلة 8 و 9 و 11 — ص 6–7 / PDF ص 58–59):**
  1. **مثال 8:** حل $(D_x - D_y^3)z = 0$.
     * المؤثر غير قابل للتحليل لأن $D_x$ من الرتبة 1 و $D_y^3$ من الرتبة 3!
     * نضع $D_x \to a_i$ و $D_y \to b_i$ في الشرط $F(a_i, b_i) = 0 \implies a_i - b_i^3 = 0 \implies \boxed{a_i = b_i^3}$.
     * نعوض عن $a_i$ بـ $b_i^3$ في المتسلسلة الأسية:
       $$\boxed{z = \sum_{i=1}^{\infty} A_i\, e^{b_i^3 x + b_i y}}$$
  2. **مثال 9:** حل $(D_x^2 + D_x + D_y)z = 0$.
     * الشرط المميز: $a_i^2 + a_i + b_i = 0 \implies \boxed{b_i = -a_i^2 - a_i = -a_i(a_i + 1)}$.
     * الحل العام: $\boxed{z = \sum_{i=1}^{\infty} A_i\, e^{a_i x - a_i(a_i + 1)y}}$.
  3. **الحالة (D) — مزيج من عامل خطي قابل للتحليل وعامل غير قابل للتحليل (مثال 11 ص 7 / PDF ص 59):**
     * في $(D_x - D_y - 1)(D_x - D_y^2)z = 0$:
       * العامل الأول خطي يعطي: $z^{(1)} = e^x \phi(y + x)$.
       * العامل الثاني $(D_x - D_y^2)$ غير قابل للتحليل ($a_i = b_i^2$) يعطي: $z^{(2)} = \sum_{i=1}^{\infty} A_i e^{b_i^2 x + b_i y}$.
       * نجمعهما معاً: $\boxed{z = e^x \phi(y + x) + \sum_{i=1}^{\infty} A_i e^{b_i^2 x + b_i y}}$!

---
### رابعاً: التكامل الخاص (P.I.) في الباب الثاني وحيلة "المرافق التفاضلي" (الباب الثاني ص 8–10 / PDF ص 60–62)

لإيجاد التكامل الخاص $z_2 = \frac{1}{F(D_x, D_y)}f(x, y)$ للمعادلات غير المتجانسة في الرتبة، نستخدم نفس قواعد اللاب 4 مع **مهارتين مهمتين جداً** في نهاية المذكرة:

#### 1. فك كثيرات الحدود بسحب أصغر مشتقة (مثال 12 ص 8 / PDF ص 60)
* **المسألة:** أوجد الحل العام لـ $(D_x^2 - D_y^2 + D_x)z = x^2 + 2y$.
  * لحساب التكامل الخاص $z_2 = \frac{1}{D_x^2 - D_y^2 + D_x}(x^2 + 2y)$، نسحب **أصغر مشتقة وهي $D_x$** من المقام:
    $$z_2 = \frac{1}{D_x\left[1 - \left(\frac{D_y^2}{D_x} - D_x\right)\right]}(x^2 + 2y) = \frac{1}{D_x}\left[1 + \left(\frac{D_y^2}{D_x} - D_x\right) + \left(\frac{D_y^2}{D_x} - D_x\right)^2 + \dots\right](x^2 + 2y)$$
  * بما أن $D_y^2(x^2 + 2y) = 0$، تختفي كل حدود $D_y^2$! ويبقى فقط:
    $$z_2 = \frac{1}{D_x}\left(1 - D_x + D_x^2\right)(x^2 + 2y) = \frac{1}{D_x}\left[(x^2 + 2y) - (2x) + 2\right] = \boxed{\frac{x^3}{3} + 2xy - x^2 + 2x}$$

#### 2. حيلة الضرب في المرافق لدوال $\sin$ و $\cos$ عندما تبقى مشتقة من الرتبة الأولى! (مثال 14 ص 9–10 / PDF ص 61–62)
* **المسألة:** أوجد الحل العام لـ $(D_x^2 - D_y)z = \sin(x - 2y)$.
  1. **الحل المكمل $z_1$:** المؤثر $(D_x^2 - D_y)$ غير قابل للتحليل ($a_i^2 - b_i = 0 \implies b_i = a_i^2$)، إذن $z_1 = \sum_{i=1}^{\infty} A_i e^{a_i x + a_i^2 y}$.
  2. **التكامل الخاص $z_2$:** هنا $f(x,y) = \sin(x - 2y)$ حيث $a = 1,\; b = -2$.
     * قاعدة الجيب تستبدل المشتقات **من الرتبة الثانية فقط** ($D_x^2 \to -a^2 = -1$ و $D_y^2 \to -b^2 = -4$)، ولكن المقام هنا هو $(D_x^2 - D_y)$ فيحتوي على $D_y$ من الرتبة الأولى!
     * **الخطوة 1:** عوض عن $D_x^2 \to -1$ أولاً:
       $$z_2 = \frac{1}{-1 - D_y}\sin(x - 2y)$$
     * **الخطوة 2 (حيلة المرافق!):** كيف نتخلص من $D_y$ في المقام ونحولها إلى $D_y^2$؟ نضرب البسط والمقام في **المرافق $(-1 + D_y)$**!
       $$z_2 = \frac{-1 + D_y}{(-1 - D_y)(-1 + D_y)}\sin(x - 2y) = \frac{-1 + D_y}{1 - D_y^2}\sin(x - 2y)$$
     * **الخطوة 3:** الآن ظهر $D_y^2$ في المقام! نعوض عنه بـ $-b^2 = -(-2)^2 = -4$:
       $$z_2 = \frac{-1 + D_y}{1 - (-4)}\sin(x - 2y) = \frac{1}{5}(-1 + D_y)\sin(x - 2y)$$
     * **الخطوة 4:** الآن $(-1 + D_y)$ في البسط يعني ببساطة: اضرب في $-1$ ثم اجمع المشتقة الجزئية بالنسبة لـ $y$ ($D_y[\sin(x-2y)] = -2\cos(x-2y)$):
       $$\boxed{z_2 = \frac{1}{5}\big[-\sin(x - 2y) - 2\cos(x - 2y)\big]}$$

---
### ⚠️ أخطاء شائعة في الإشارات يجب الحذر منها
1. **نسيان إشارة السالب في الأس $-\frac{c}{a}x$:** إذا كان $c$ موجباً (مثل $+1$)، يكون الأس سالباً ($-x/a$)؛ وإذا كان $c$ سالباً (مثل $-2$)، يصبح الأس موجباً ($+2x/a$).
2. **نسيان إشارة السالب في $(ay - bx)$:** القانون يحتوي على طرح $-bx$! فإذا كان $b = -3$، تصبح $ay - (-3)x = ay + 3x$.

</div>


<div dir="rtl" style="text-align:right">

---
## 4. 📝 تحدي الورقة والقلم 1 (المثال 5 من الباب الثاني ص 4 / PDF ص 56)

**المسألة:** أوجد الحل العام للمعادلة التفاضلية الجزئية القابلة للتحليل:
$$(2 D_x - 3 D_y + 1)(D_x + 2 D_y - 2)z = 0$$

اكتب صيغة الحل العام بدلالة `x, y` والدالتين الاختياريتين `Phi1(...)` و `Phi2(...)` والدالة الأسية `exp(...)`:

<details>
<summary>💡 <b>تلميح 1 (العامل الأول)</b></summary>

للعامل $(2 D_x - 3 D_y + 1)$: $a_1=2, b_1=-3, c_1=1 \implies \text{exp}(-x/2)\text{*Phi1}(2y + 3x)$.
</details>

<details>
<summary>💡 <b>تلميح 2 (العامل الثاني والجمع)</b></summary>

للعامل $(D_x + 2 D_y - 2)$: $a_2=1, b_2=2, c_2=-2 \implies \text{exp}(2x)\text{*Phi2}(y - 2x)$. اجمع الحدين معاً.
</details>

</div>


In [ ]:
# خلية تقييم: تحدي الورقة 1
### START CODE HERE ###
student_solution = "exp(-x/2)*Phi1(2y + 3x) + exp(2x)*Phi2(y - 2x)"
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab5.check_paper_challenge_1(student_solution)


<div dir="rtl" style="text-align:right">

---
## 5. 🚀 تحدي الورقة والقلم 2 (حساب معدل الاضمحلال الأسي $k = -c/a$)

**المسألة:** لنفرض أن نمطاً موجياً يخضع للمعادلة العاملية الخطية:
$$(D_x + 2 D_y + 1)z = 0$$

**على ورقتك:**
1. استخرج المعاملات $a, b, c$ من العامل $(a D_x + b D_y + c)$.
2. في الحل العام $z = e^{k x} \Phi(a y - b x)$، يُعطى معدل التخميد الأسي بالعلاقة $k = -\frac{c}{a}$.
3. احسب القيمة العددية لـ $k$ وأدخلها أدناه:

<details>
<summary>💡 <b>تلميح (اضغط هنا إذا احتجت مساعدة)</b></summary>

معامل $D_x$ هو $a = 1$، والحد الثابت هو $c = 1$. إذن $k = -c/a = -1/1$.
</details>

</div>


In [ ]:
# خلية تقييم: معدل الاضمحلال الأسي
### START CODE HERE ###
damping_rate = -1.0
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab5.check_paper_challenge_2(damping_rate)


<div dir="rtl" style="text-align:right">

---
## 6. 💻 التطبيق البرمجي ببايثون: محلل معاملات العوامل الخطية (`parse_linear_factor`)

نريد كتابة دالة برمجية تستقبل نص عامل خطي مثل `"2*D_x - 3*D_y - 5"` وتستخرج منه المعاملات العددية الثلاثة `(a, b, c)` تلقائياً لحساب معدل الاضمحلال الأسي $-c/a$ وزاوية الانتشار $(a, -b)$.

</div>


In [ ]:
# دالة تقييم برمجية: parse_linear_factor
def parse_linear_factor(factor_str: str) -> tuple[float, float, float]:
    # تستخرج المعاملات (a, b, c) من نص العامل الخطي "a*D_x + b*D_y + c"
    ### START CODE HERE ###
    import re
    ma = re.search(r'([+-]?\s*\d*)\s*\*?\s*D_x', factor_str)
    mb = re.search(r'([+-]?\s*\d*)\s*\*?\s*D_y', factor_str)
    mc = re.search(r'([+-]?\s*\d+)\s*$', factor_str.replace(" ", ""))
    
    def to_float(val, default):
        if not val or val.strip() in ['+', '']: return default
        if val.strip() == '-': return -default
        return float(val.replace(" ", ""))

    a = to_float(ma.group(1), 1.0) if ma else 0.0
    b = to_float(mb.group(1), 1.0) if mb else 0.0
    c = float(mc.group(1)) if mc else 0.0
    ### END CODE HERE ###
    return a, b, c

# تشغيل الاختبار الآلي:
test_lab5.test_linear_factor_evaluator(parse_linear_factor)


<div dir="rtl" style="text-align:right">

---
## 7. 🎨 الرؤية البصرية والربط المباشر بملف `rover_path_finder/planners.py`

شغّل الخلية التالية لترى:
1. **على اليسار:** السطح الأسي المتلاشي $z = e^{-0.5x}\sin(y - 2x)$ الناتج عن العامل الخطي $(2D_x + D_y + 1)z = 0$.
2. **على اليمين:** مقارنة حقيقية من كود المشروع [`rover_path_finder/planners.py`](rover_path_finder/planners.py) بين دالة `dirichlet_potential` (التي ينهار فيها انحدار الجهد أسياً بمعدل $e^{-\pi x / w}$ داخل ممر ضيق فتتوقف المركبة!) ودالة `neumann_potential` (التي تعزل جدران الممر $\partial\phi/\partial n = 0$ فينحدر الجهد خطياً بسلاسة من البداية إلى الهدف!).

</div>


In [ ]:
from rover_path_finder.planners import dirichlet_potential, neumann_potential

# 1. السطح التحليلي للعامل الخطي (2 D_x + D_y + 1)z = 0
x = np.linspace(0, 5, 50)
y = np.linspace(-3, 3, 50)
X, Y = np.meshgrid(x, y)
Z = np.exp(-0.5 * X) * np.sin(Y - 2.0 * X)

# 2. تجربة ممر ضيق بين جدارين باستخدام دوال المشروع الفعلية
corridor_hazard = np.zeros((40, 60), dtype=bool)
corridor_hazard[:16, 15:45] = True   # الجدار العلوي للممر الضيق
corridor_hazard[24:, 15:45] = True   # الجدار السفلي للممر الضيق (عرض الممر w = 8 بكسل)
start_pt, goal_pt = (8, 20), (52, 20)

phi_dir, _, _ = dirichlet_potential(corridor_hazard, goal=goal_pt, tol=1e-8)
phi_neu, _, _ = neumann_potential(corridor_hazard, start=start_pt, goal=goal_pt, tol=1e-6)

fig = plt.figure(figsize=(14, 5.5))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
surf = ax1.plot_surface(X, Y, Z, cmap='plasma', alpha=0.88)
ax1.set_title(r"1. Linear Factor Solution: $z = e^{-0.5x}\sin(y - 2x)$", fontsize=11)
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.set_zlabel("z")
plt.colorbar(surf, ax=ax1, shrink=0.55)

ax2 = fig.add_subplot(1, 2, 2)
center_x = np.arange(8, 53)
ax2.plot(center_x, 1.0 - phi_dir[20, 8:53], 'r--', linewidth=2.5, label=r"1 - dirichlet_potential (Exponential Decay $e^{-\pi x/w}$)")
ax2.plot(center_x, phi_neu[20, 8:53], 'b-', linewidth=2.5, label="neumann_potential (Smooth Linear Gradient)")
ax2.set_yscale('log')
ax2.set_title("2. Real Engine: Narrow Corridor Gradient Decay (Log Scale)", fontsize=11)
ax2.set_xlabel("Position along narrow corridor (x)")
ax2.set_ylabel("Potential Drop (log scale)")
ax2.grid(True, which="both", alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

print("📌 ملخص اللاب 5:")
print("  1) العامل الخطي (a*D_x + b*D_y + c)z = 0 ينتج حلاً مضروباً في غلاف أسي exp(-(c/a)*x).")
print("  2) في الممرات الضيقة، شرط ديريكليه على الجدران يخلق تلاشيًا أسيًا exp(-pi*x/w) يوقف المركبة.")
print("  3) لهذا السبب يستخدم محرك الملاحة في اللاب 6 شروط نيومان العازلة (neumann_potential) بدلاً من ديريكليه!")
